In [ ]:
#1.1
# Import required libraries
import pandas as pd
import json
import re
import html

# Load the JSON dataset
with open('research_abstracts.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Convert JSON data into a DataFrame
df = pd.DataFrame(data)

# Display the first five records
print(df.head())

# Check dataset information
print(df.info())

# Check missing values
print(df.isnull().sum())

# Remove records with missing abstracts or labels
df = df.dropna(subset=['Abstract', 'Label'])

# Remove duplicate abstracts
df = df.drop_duplicates(subset=['Abstract', 'Label'])

# Text cleaning function
def clean_text(text):

    # Convert text to string
    text = str(text)

    # Remove HTML entities
    text = html.unescape(text)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Convert text to lowercase
    text = text.lower()

    # Remove numbers and special characters
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Remove unnecessary whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply text cleaning
df['Clean_Abstract'] = df['Abstract'].apply(clean_text)

# Remove DOI because it is not required for text classification
df = df.drop(columns=['DOI'])

# Display cleaned data
print(df.head())

# Check final dataset
print("Dataset shape:", df.shape)
print(df['Label'].value_counts())


                          DOI  \
0        10.1155/2021/1071145   
1  10.1007/s43546-022-00384-2   
2            10.1002/for.2667   
3  10.1007/s42973-021-00091-x   
4   10.1007/s40258-012-0003-z   

                                            Abstract      Label  
0  Forward-looking forecasting of the inflation r...         ai  
1  Our study examined the disaggregation of infla...         ai  
2  This paper develops a dynamic factor model tha...  classical  
3  In this paper, I use a simple SIR Macro model ...  classical  
4  BackgroundGame theory is useful for identifyin...  classical  
<class 'pandas.core.frame.DataFrame'>
Index: 1138 entries, 0 to 1137
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   DOI       1134 non-null   object
 1   Abstract  1138 non-null   object
 2   Label     1138 non-null   object
dtypes: object(3)
memory usage: 35.6+ KB
None
DOI         4
Abstract    0
Label       0
dtype: int64
             

In [ ]:
#1.2
from sklearn.feature_extraction.text import TfidfVectorizer

# Select input features and target variable
X = df['Clean_Abstract']
y = df['Label']

# Convert text into numerical features using TF-IDF
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

X_tfidf = tfidf.fit_transform(X)

# Display feature information
print("TF-IDF feature matrix:", X_tfidf.shape)
print("Number of extracted features:", len(tfidf.get_feature_names_out()))


TF-IDF feature matrix: (1125, 5000)
Number of extracted features: 5000


In [ ]:
#1.3
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix

# Select features and target
X = df['Clean_Abstract']
y = df['Label']

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create baseline model
model = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',
        max_features=5000,
        ngram_range=(1, 2),
        min_df=2
    )),
    ('classifier', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])

# Train the model
model.fit(X_train, y_train)

# Make predictions
train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)

# Evaluate training performance
print("TRAINING PERFORMANCE")
print(classification_report(y_train, train_predictions))

# Evaluate testing performance
print("TESTING PERFORMANCE")
print(classification_report(y_test, test_predictions))

# Calculate accuracy
print("Training Accuracy:",
      accuracy_score(y_train, train_predictions))

print("Testing Accuracy:",
      accuracy_score(y_test, test_predictions))

# Display confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_predictions))


TRAINING PERFORMANCE
              precision    recall  f1-score   support

          ai       0.90      0.99      0.94       188
   classical       1.00      0.97      0.98       712

    accuracy                           0.98       900
   macro avg       0.95      0.98      0.96       900
weighted avg       0.98      0.98      0.98       900

TESTING PERFORMANCE
              precision    recall  f1-score   support

          ai       0.74      0.79      0.76        47
   classical       0.94      0.93      0.93       178

    accuracy                           0.90       225
   macro avg       0.84      0.86      0.85       225
weighted avg       0.90      0.90      0.90       225

Training Accuracy: 0.9755555555555555
Testing Accuracy: 0.8977777777777778
Confusion Matrix:
[[ 37  10]
 [ 13 165]]


In [ ]:
#2.1
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Features and target
X = df['Clean_Abstract']
y = df['Label']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Define four machine learning models
models = {
    'Multinomial Naive Bayes': MultinomialNB(),
    'Complement Naive Bayes': ComplementNB(),
    'Linear SVC': LinearSVC(class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
}

# Store model results and trained pipelines
results = []
trained_models = {}

for name, classifier in models.items():

    model = Pipeline([
        ('tfidf', TfidfVectorizer(
            stop_words='english',
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2
        )),
        ('classifier', classifier)
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(
        y_test, predictions, average='weighted', zero_division=0
    )
    recall = recall_score(
        y_test, predictions, average='weighted', zero_division=0
    )
    f1 = f1_score(
        y_test, predictions, average='weighted', zero_division=0
    )

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    })

    trained_models[name] = model

# Display comparison
results_df = pd.DataFrame(results)
print(results_df.sort_values(by='F1 Score', ascending=False))

# Display detailed report for each model
for name, model in trained_models.items():
    print("\n", name)
    print(classification_report(y_test, model.predict(X_test)))


                     Model  Accuracy  Precision    Recall  F1 Score
3            Random Forest  0.915556   0.923700  0.915556  0.906999
2               Linear SVC  0.906667   0.904012  0.906667  0.904691
1   Complement Naive Bayes  0.897778   0.894027  0.897778  0.892536
0  Multinomial Naive Bayes  0.844444   0.870005  0.844444  0.805267

 Multinomial Naive Bayes
              precision    recall  f1-score   support

          ai       1.00      0.26      0.41        47
   classical       0.84      1.00      0.91       178

    accuracy                           0.84       225
   macro avg       0.92      0.63      0.66       225
weighted avg       0.87      0.84      0.81       225


 Complement Naive Bayes
              precision    recall  f1-score   support

          ai       0.83      0.64      0.72        47
   classical       0.91      0.97      0.94       178

    accuracy                           0.90       225
   macro avg       0.87      0.80      0.83       225
weighted a

In [ ]:
#2.2
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Create a pipeline
svc_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('classifier', LinearSVC(class_weight='balanced', random_state=42))
])

# Parameters to test
parameters = {
    'tfidf__max_features': [3000, 5000, 7000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'classifier__C': [0.1, 1, 2, 5]
}

# Perform grid search using cross-validation
grid_search = GridSearchCV(
    svc_pipeline,
    parameters,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

# Train and tune the model
grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_

# Test the tuned model
tuned_predictions = best_model.predict(X_test)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-validation F1:", grid_search.best_score_)

print("Tuned Model Accuracy:",
      accuracy_score(y_test, tuned_predictions))

print("Tuned Model F1 Score:",
      f1_score(y_test, tuned_predictions, average='weighted'))

print("\nClassification Report:")
print(classification_report(y_test, tuned_predictions))

# Compare with the original baseline
print("Baseline Accuracy: 0.8978")
print("Tuned Model Accuracy:",
      accuracy_score(y_test, tuned_predictions))


Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Parameters: {'classifier__C': 0.1, 'tfidf__max_features': 3000, 'tfidf__ngram_range': (1, 2)}
Best Cross-validation F1: 0.8368406565283376
Tuned Model Accuracy: 0.9022222222222223
Tuned Model F1 Score: 0.902964532328939

Classification Report:
              precision    recall  f1-score   support

          ai       0.76      0.79      0.77        47
   classical       0.94      0.93      0.94       178

    accuracy                           0.90       225
   macro avg       0.85      0.86      0.85       225
weighted avg       0.90      0.90      0.90       225

Baseline Accuracy: 0.8978
Tuned Model Accuracy: 0.9022222222222223


In [ ]:
#2.3
import joblib

# Save the best model from Question 2.2
joblib.dump(best_model, "best_research_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [8]:
import gradio as gr
import joblib

# Load the trained model
model = joblib.load("best_research_model.pkl")

# Function to classify the abstract
def classify_abstract(abstract):

    if not abstract.strip():
        return "Please enter a research abstract."

    prediction = model.predict([abstract])[0]

    if prediction == "ai":
        return "The abstract is classified as: artificial intelligence"
    else:
        return "The abstract is classified as: classical research"


# Create the interface
with gr.Blocks(title="Researchly Research Abstract Classifier") as demo:

    gr.Markdown("# Researchly Research Abstract Classifier")

    gr.Markdown(
        "This app classifies research abstracts as either "
        "AI-related or classical research."
    )

    abstract_input = gr.Textbox(
        label="Enter research abstract:",
        lines=8,
        placeholder="Paste your research abstract here..."
    )

    classify_button = gr.Button("Classify Abstract")

    gr.Markdown("## Prediction")

    prediction_output = gr.Textbox(
        label="",
        interactive=False
    )

    classify_button.click(
        fn=classify_abstract,
        inputs=abstract_input,
        outputs=prediction_output
    )

# Display the web application
demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
